# Chapter 10 — Classical Classifiers
**Track:** ML from Scratch · California Housing — high-value district classifier (interpretable models)

## Core Idea

**Decision Tree:** partition feature space with binary `if/else` rules. Inspectable, fast, naturally non-linear.  
**KNN:** predict by majority vote among the $k$ most similar training examples. No training step.

```
Decision Tree:   if MedInc > 3.2 and Latitude < 36.1 → High-Value   (human-readable)
KNN (k=5):       5 nearest neighbours are all High-Value → High-Value (geometric)
```

Both overfit in characteristic ways: DT → too deep (one leaf per point); KNN → k=1 (memorises noise).  
Primary dials: **`max_depth`** (DT) and **`k`** (KNN).

## Running Example

Same **California Housing** dataset, same `high_value` binary target as Ch.2 and Ch.9.  
The task: build an auditable classifier the compliance team can explain.

This lets us compare Decision Tree and KNN directly against Logistic Regression (Ch.2) and the neural network (Ch.4) on the same problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# ── Data ──────────────────────────────────────────────────────────────────────
data         = fetch_california_housing()
X, y_reg     = data.data, data.target
feature_names = list(data.feature_names)

y = (y_reg > np.median(y_reg)).astype(int)   # high-value binary target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"Class balance: {y_train.mean():.3f} positive (median split → ~50/50)")
print(f"Features: {feature_names}")

## Gini Impurity — The Split Criterion

$$\text{Gini}(S) = 1 - \sum_c p_c^2$$

A node is split at the $(f, t)$ pair that maximises **information gain**:

$$\text{IG}(S, f, t) = \text{Gini}(S) - \frac{|S_L|}{|S|}\,\text{Gini}(S_L) - \frac{|S_R|}{|S|}\,\text{Gini}(S_R)$$

In [ ]:
def gini(p_pos):
    """Gini impurity for a binary node with fraction p_pos positives."""
    return 1 - p_pos**2 - (1 - p_pos)**2

def information_gain(p_parent, n_left, n_right, p_left, p_right):
    """Information gain of a binary split."""
    n_total = n_left + n_right
    return (gini(p_parent)
            - (n_left / n_total)  * gini(p_left)
            - (n_right / n_total) * gini(p_right))

# Example: root node for our dataset (≈50/50 split)
p_root  = y_train.mean()
g_root  = gini(p_root)
print(f"Root Gini:  {g_root:.4f}  (p_pos={p_root:.3f}, maximally mixed → close to 0.5)")

# Simulate a good split: left child 85% positive, right child 20% positive
n_left, n_right = 800, 1500
p_left, p_right = 0.85, 0.20
ig = information_gain(p_root, n_left, n_right, p_left, p_right)
print(f"Left  child Gini: {gini(p_left):.4f}  (n={n_left}, p={p_left})")
print(f"Right child Gini: {gini(p_right):.4f}  (n={n_right}, p={p_right})")
print(f"Information Gain: {ig:.4f}")

# Plot Gini as a function of class balance
p_vals = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(p_vals, [gini(p) for p in p_vals], color='steelblue', linewidth=2.5)
ax.axvline(0.5, color='coral', linestyle='--', alpha=0.7, label='Max impurity at p=0.5')
ax.set_xlabel('Fraction of positive class (p)')
ax.set_ylabel('Gini impurity')
ax.set_title('Gini Impurity vs Class Balance', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Decision Tree — Train and Visualise

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_sc, y_train)   # DT doesn't need scaling but we pass scaled data for fair comparison

y_pred_dt = dt.predict(X_test_sc)
print(f"Decision Tree (max_depth=5)")
print(f"  Train accuracy: {accuracy_score(y_train, dt.predict(X_train_sc)):.4f}")
print(f"  Test  accuracy: {accuracy_score(y_test,  y_pred_dt):.4f}")
print(f"  Test  F1:       {f1_score(y_test, y_pred_dt):.4f}")
print(f"  Test  AUC-ROC:  {roc_auc_score(y_test, dt.predict_proba(X_test_sc)[:, 1]):.4f}")

In [ ]:
# Human-readable rules (first 3 levels)
print("Decision rules (depth ≤ 3):")
print(export_text(dt, feature_names=feature_names, max_depth=3))

In [ ]:
# Graphical tree plot (top 3 levels)
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt, max_depth=3, feature_names=feature_names,
          class_names=['Non-HV', 'High-Value'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree — Top 3 Levels (max_depth=5 trained)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Feature Importances

Importance of feature $j$ = total impurity reduction at all splits on $j$, weighted by the fraction of samples reaching each split node.  
Normalised to sum to 1.

In [ ]:
importances = dt.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh([feature_names[i] for i in sorted_idx[::-1]],
        importances[sorted_idx[::-1]],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Feature importance (Gini reduction)')
ax.set_title('Decision Tree Feature Importances', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Feature importances:")
for i in sorted_idx:
    bar = '█' * int(importances[i] * 60)
    print(f"  {feature_names[i]:12s}: {importances[i]:.4f}  {bar}")

## Hyperparameter Dial — `max_depth`

Too shallow → underfit. Too deep → memorise training set (high variance).  
Plot train vs test F1 as `max_depth` increases to find the bias-variance sweet spot.

In [ ]:
depths = range(1, 26)
train_f1s, test_f1s = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train_sc, y_train)
    train_f1s.append(f1_score(y_train, m.predict(X_train_sc)))
    test_f1s.append(f1_score(y_test,   m.predict(X_test_sc)))

best_depth = list(depths)[np.argmax(test_f1s)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(depths, train_f1s, label='Train F1', color='coral',     linewidth=2)
ax.plot(depths, test_f1s,  label='Test F1',  color='steelblue', linewidth=2)
ax.axvline(best_depth, color='black', linestyle='--', alpha=0.6,
           label=f'Best test depth = {best_depth}  (F1={max(test_f1s):.4f})')
ax.set_xlabel('max_depth'); ax.set_ylabel('F1 score')
ax.set_title('Decision Tree — Train vs Test F1 by Depth', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal max_depth: {best_depth}  (test F1={max(test_f1s):.4f})")
print(f"Fully grown (depth=None) train F1: ~1.0  — perfect memorisation")

## KNN — Distance and Voting

$$d(\mathbf{x}_q, \mathbf{x}_i) = \sqrt{\sum_j (x_{q,j} - x_{i,j})^2}$$

$\hat{y}_q = \text{mode}\{y_i : i \in \mathcal{N}_k(\mathbf{x}_q)\}$

**Critical:** KNN distances are meaningless on unscaled features — always standardise first.

In [ ]:
# KNN with k=11 (odd to avoid ties, ~sqrt(n_train) ≈ 128 but start smaller)
knn = KNeighborsClassifier(n_neighbors=11, metric='euclidean')
knn.fit(X_train_sc, y_train)

y_pred_knn = knn.predict(X_test_sc)
print(f"KNN (k=11, scaled)")
print(f"  Test accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"  Test F1:       {f1_score(y_test, y_pred_knn):.4f}")
print(f"  Test AUC-ROC:  {roc_auc_score(y_test, knn.predict_proba(X_test_sc)[:, 1]):.4f}")

## What Can Go Wrong — KNN on Unscaled Features

In [ ]:
# KNN on raw (unscaled) features — distance dominated by largest-range features
knn_unscaled = KNeighborsClassifier(n_neighbors=11, metric='euclidean')
knn_unscaled.fit(X_train, y_train)
y_pred_knn_raw = knn_unscaled.predict(X_test)

f1_scaled   = f1_score(y_test, y_pred_knn)
f1_unscaled = f1_score(y_test, y_pred_knn_raw)

print(f"KNN unscaled features → F1: {f1_unscaled:.4f}")
print(f"KNN   scaled features → F1: {f1_scaled:.4f}")
print(f"Performance drop from not scaling: {(f1_scaled - f1_unscaled):.4f} F1 points")

print(f"\nFeature ranges (raw):")
for name, lo, hi in zip(feature_names, X.min(axis=0), X.max(axis=0)):
    print(f"  {name:12s}: [{lo:.2f}, {hi:.2f}]  range={hi-lo:.2f}")
print("\nThe feature with the largest range dominates Euclidean distance.")

## Hyperparameter Dial — $k$ in KNN

In [ ]:
k_values = range(1, 102, 2)   # odd k values to avoid ties
knn_f1s  = []

for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_sc, y_train)
    knn_f1s.append(f1_score(y_test, m.predict(X_test_sc)))

best_k     = list(k_values)[np.argmax(knn_f1s)]
best_knn_f1 = max(knn_f1s)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(k_values), knn_f1s, color='steelblue', linewidth=2)
ax.axvline(best_k, color='coral', linestyle='--', alpha=0.8,
           label=f'Best k={best_k}  (F1={best_knn_f1:.4f})')
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('Test F1')
ax.set_title('KNN — Test F1 vs k', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"k=1   F1: {knn_f1s[0]:.4f}  (memorises training set — high variance)")
print(f"k={best_k:3d} F1: {best_knn_f1:.4f}  (optimal)")
print(f"k=101 F1: {knn_f1s[-1]:.4f}  (too smooth — high bias)")

## Model Comparison — DT vs KNN vs Logistic Regression

In [ ]:
# Retrain best DT and KNN; add logistic regression as baseline
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_sc, y_train)
dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=42).fit(X_train_sc, y_train)
knn_best = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_sc, y_train)

models = {
    'Logistic Regression': lr,
    f'Decision Tree (depth={best_depth})': dt_best,
    f'KNN (k={best_k})': knn_best,
}

print(f"{'Model':<35} {'Accuracy':>10} {'F1':>10} {'AUC-ROC':>10}")
print("-" * 70)
for name, model in models.items():
    y_p  = model.predict(X_test_sc)
    y_pr = model.predict_proba(X_test_sc)[:, 1]
    print(f"{name:<35} {accuracy_score(y_test, y_p):>10.4f}"
          f" {f1_score(y_test, y_p):>10.4f}"
          f" {roc_auc_score(y_test, y_pr):>10.4f}")

print("\nKey trade-off:")
print("  Decision Tree → interpretable rules, business can audit decisions")
print("  KNN → no assumptions about decision boundary shape")
print("  Logistic Regression → fast, probabilistic, linear boundary")

## Exercises

1. **Post-pruning:** train a Decision Tree to full depth (`max_depth=None`), then use `cost_complexity_pruning_path` to find the best `ccp_alpha` via cross-validation. How does it compare to `max_depth` tuning?
2. **Weighted KNN:** switch `KNeighborsClassifier(weights='distance')`. Does giving closer neighbours more vote weight improve F1 vs uniform weights?
3. **Feature importance stability:** train 10 Decision Trees on random 80% subsamples of training data. Print the feature importance ranking for each. How stable is the ordering? (Preview of why Random Forest importances are better.)

In [ ]:
# Exercise 1: cost-complexity pruning
# TODO: dt_full = DecisionTreeClassifier(random_state=42).fit(X_train_sc, y_train)
# path = dt_full.cost_complexity_pruning_path(X_train_sc, y_train)
# Try several ccp_alpha values. Plot test F1 vs ccp_alpha.

# Exercise 2: weighted KNN
# TODO: knn_dist = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
# Compare F1 of uniform vs distance weighting on the test set.

# Exercise 3: feature importance stability
# TODO: for seed in range(10): train DT on 80% subsample, record feature_importances_
# Print ranking table. Do the top 2 features stay the same across seeds?